In [1]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from transformers import ASTForAudioClassification
from transformers import DefaultDataCollator
#from datasets import load_metric
import evaluate
from transformers import Trainer, TrainingArguments
import os
from transformers import EarlyStoppingCallback

c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0511 14:11:32.427000 16212 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
pip install transformers==4.39.3

In [2]:
data_path = Path(r"C:\Users\Kochana\projects\genres\data\gtzan\gtzan.npz")
data = np.load(data_path)
lst = data.files

In [13]:
lst[0][6:][:-12]

'blues'

In [10]:
data[lst[0]]

array([[4.246  , 3.527  , 4.26   , ..., 1.512  , 1.492  , 1.501  ],
       [4.945  , 4.195  , 4.457  , ..., 0.2866 , 0.1768 , 0.1576 ],
       [4.66   , 4.37   , 4.113  , ..., 0.2715 , 0.07697, 0.05035],
       ...,
       [5.44   , 4.523  , 4.582  , ..., 0.21   , 0.0781 , 0.04965],
       [6.016  , 6.496  , 6.945  , ..., 0.4546 , 0.1697 , 0.06555],
       [6.26   , 6.312  , 6.8    , ..., 1.133  , 1.117  , 1.133  ]],
      shape=(1501, 128), dtype=float16)

In [3]:
tracks_path = []
labels = []
#data_path = Path(r"/content/drive/MyDrive/data/gtzan_old")
for path in lst:
    #path_file = os.path.join(data_path, file, "track.npy")
    tracks_path.append(path)
    labels.append(path[6:][:-12])
le = LabelEncoder()
encoded_labels = le.fit_transform(labels)
train, validation, train_labels, val_labels = train_test_split(
        tracks_path, encoded_labels, test_size=0.2, stratify=encoded_labels, random_state=42)


In [25]:
data[train[0]]

array([[4.25   , 5.08   , 5.64   , ..., 2.627  , 2.62   , 2.623  ],
       [4.598  , 4.812  , 5.45   , ..., 0.4714 , 0.4685 , 0.4678 ],
       [4.08   , 4.4    , 4.816  , ..., 0.0981 , 0.0616 , 0.074  ],
       ...,
       [3.914  , 3.848  , 3.924  , ..., 0.0744 , 0.05515, 0.054  ],
       [4.09   , 2.87   , 2.686  , ..., 0.0705 , 0.05994, 0.06647],
       [4.38   , 4.383  , 3.965  , ..., 1.419  , 1.455  , 1.48   ]],
      shape=(1501, 128), dtype=float16)

In [9]:
import wandb
wandb.init(project="ast_model", name="wadims_pc_try_2_early_stop")

In [11]:
class GTZANSpectrogramDataset(Dataset):
    def __init__(self, path, labels):
        self.paths = path
        self.labels = labels
        self.max_time = 1020
        self.data = data
        
    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        #spec = self.spectrograms[idx]  # shape: (128, time)
        spec = data[self.paths[idx]]
        #print(spec.shape)
        #spec = spec[None, :, :]  # add channel dim
        if spec.ndim == 3 and spec.shape[0] == 1:
            spec = spec.squeeze(0)
        spec = spec[:self.max_time, :]
        spec = torch.tensor(spec, dtype=torch.float32)

        #spec = spec.unsqueeze(0)
        label = self.labels[idx]
        return {"input_values": spec, "labels": label}

model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,  # GTZAN has 10 genres
    ignore_mismatched_sizes=True  # allows adjusting output layer
)
data_collator = DefaultDataCollator()
#metric = load_metric("accuracy")
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return metric.compute(predictions=predictions, references=labels)
training_args = TrainingArguments(
    output_dir="./ast-gtzan",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=3e-5,
    num_train_epochs=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    gradient_accumulation_steps=8,
    greater_is_better=True,
    report_to="wandb",
)

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
train_dataset = GTZANSpectrogramDataset(train, train_labels)
val_dataset = GTZANSpectrogramDataset(validation, val_labels)
trainer = Trainer(
    model = model,
    args= training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=None,  # AST doesn't use a tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience =4)]
)
trainer.train()

c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\accelerate\accelerator.py:463: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
  2%|▏         | 50/2500 [00:49<35:17,  1.16it/s]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/gene

{'eval_loss': 1.0123720169067383, 'eval_accuracy': 0.635, 'eval_runtime': 4.4551, 'eval_samples_per_second': 44.892, 'eval_steps_per_second': 22.446, 'epoch': 1.0}


  4%|▍         | 100/2500 [01:38<34:25,  1.16it/s]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 1024}


{'eval_loss': 0.6520993709564209, 'eval_accuracy': 0.81, 'eval_runtime': 4.4355, 'eval_samples_per_second': 45.09, 'eval_steps_per_second': 22.545, 'epoch': 2.0}


  6%|▌         | 150/2500 [02:27<33:45,  1.16it/s]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 1024}


...

 26%|██▌       | 650/2500 [10:38<26:35,  1.16it/s]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 1024}


{'eval_loss': 0.6857988238334656, 'eval_accuracy': 0.84, 'eval_runtime': 4.4449, 'eval_samples_per_second': 44.995, 'eval_steps_per_second': 22.498, 'epoch': 13.0}


 26%|██▌       | 650/2500 [10:39<30:20,  1.02it/s]

{'train_runtime': 639.7502, 'train_samples_per_second': 62.446, 'train_steps_per_second': 3.908, 'train_loss': 0.29840478383577784, 'epoch': 13.0}


TrainOutput(global_step=650, training_loss=0.29840478383577784, metrics={'train_runtime': 639.7502, 'train_samples_per_second': 62.446, 'train_steps_per_second': 3.908, 'train_loss': 0.29840478383577784, 'epoch': 13.0})

In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_summary(device=0))

True
NVIDIA GeForce RTX 4070 SUPER
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 1            |        cudaMalloc retries: 2         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  26258 MiB |  26258 MiB |  41196 MiB |  14937 MiB |
|       from large pool |  26253 MiB |  26253 MiB |  41158 MiB |  14904 MiB |
|       from small pool |      4 MiB |      5 MiB |     37 MiB |     33 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  26258 MiB |  26258 MiB |  41196 MiB |  14937 MiB |
|       from large pool |  26

In [6]:
print(torch.cuda.is_available())

True


In [ ]:
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [17]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


^C
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

^C
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Zugriff verweigert: 'C:\\Users\\Kochana\\projects\\genres\\ast_venv\\Lib\\site-packages\\~orch\\lib\\asmjit.dll'
Check the permissions.


[notice] A new release of pip available: 22.3 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu126
  Obtaining dependency information for torchvision from https://download.pytorch.org/whl/cu126/torchvision-0.22.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torchaudio from https://download.pytorch.org/whl/cu126/torchaudio-2.7.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torch from https://download.pytorch.org/whl/cu126/torch-2.7.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for pillow!=8.3.*,>=5.3.0 from https://download.pytorch.org/whl/pillow-11.0.0-cp311-cp311-win_amd64.whl.metadata
  Using cached https://download.pytorch.org/whl/pillow-11.0.0-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
   ---------------------------------------- 6.3/6.3 MB 6.6 MB/s eta 0:00:00
   ---------------------------------------- 2.8/2.8 GB 994.8 kB/s eta 0:00:00
   ---------------------------------------- 4.2/4.2 MB 7.1 MB/s eta 0:00:00
  

In [1]:
import torch
print(torch.__version__)

2.7.0+cpu


In [25]:
pip install accelerate==0.28.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip
